## STEP 5 - Apply window functions for advanced analytics

In [0]:
#Rank products by daily sales using partitionBy sale_date
products_by_daily_sales_df = spark.sql("""
WITH daily_sales AS (
  SELECT a.product_id, a.product_name, a.category, a.brand, a.unit_price, a.launch_date, a.is_active, b.sale_date, SUM(b.total_amount) AS daily_sale_sum
  FROM silver_db.cleand_products_data a
  JOIN silver_db.cleand_sales_data b ON a.product_id = b.product_id
  GROUP BY a.product_id, a.product_name, a.category, a.brand, a.unit_price, a.launch_date, a.is_active, b.sale_date
)
SELECT *, dense_rank() OVER (PARTITION BY sale_date ORDER BY daily_sale_sum DESC) AS product_rank
FROM daily_sales
""")
display(products_by_daily_sales_df)


In [0]:
products_by_daily_sales_df.write.format("delta").mode("overwrite").saveAsTable("gold_db.products_by_daily_sales")

In [0]:
#Compute running revenue totals per store over time

running_revenue_totals_per_store = spark.sql("""
WITH daily_revenue AS (
    SELECT 
        a.store_id,
        a.store_name,
        a.city,
        a.state,
        a.store_type,
        a.opened_date,
        b.sale_date,
        SUM(b.price_int * b.quantity) AS daily_revenue
    FROM silver_db.cleand_stores_data a
    JOIN silver_db.cleand_sales_data b
        ON a.store_id = b.store_id
    GROUP BY
        a.store_id,
        a.store_name,
        a.city,
        a.state,
        a.store_type,
        a.opened_date,
        b.sale_date
)

SELECT *,
       SUM(daily_revenue) OVER (
           PARTITION BY store_id
           ORDER BY sale_date
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
       ) AS running_total_revenue
FROM daily_revenue
ORDER BY store_id, sale_date
""")

display(running_revenue_totals_per_store)


In [0]:
running_revenue_totals_per_store.write.format("Delta").mode("Overwrite").saveAsTable("gold_db.running_revenue_totals_per_store")

In [0]:
#Calculate cumulative sales for each product
cumulative_sales_df = spark.sql("""
SELECT 
    a.product_id,
    a.product_name,
    a.category,
    a.brand,
    a.unit_price,
    a.launch_date,
    b.sale_id,
    b.sale_date,
    b.price_int,
    b.quantity_int,
    SUM(b.price_int) OVER (
        PARTITION BY a.product_id
        ORDER BY b.sale_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cumulative_sales
FROM silver_db.cleand_products_data a
LEFT JOIN silver_db.cleand_sales_data b
    ON a.product_id = b.product_id
""")
display(cumulative_sales_df)


In [0]:
#Use rank(), row_number(), and sum() over window specifications
window_spec_df = spark.sql("""
SELECT 
    product_id,
    sale_date,
    total_amount,
    SUM(total_amount) OVER (PARTITION BY product_id ORDER BY sale_date) AS running_total,
    RANK() OVER (PARTITION BY sale_date ORDER BY total_amount DESC) AS sales_rank,
    ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY sale_date) AS sale_row_number
FROM silver_db.cleand_sales_data
""")
display(window_spec_df)

In [0]:
window_spec_df.write.format("delta").mode("Overwrite").saveAsTable("gold_db.window_spec")